# SurfaceTarget-CRC — phase 01 — patch 1 10

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.10
#
# RNA-seq Sample and File Selection Audit
#
# Objectives:
# 1. Match Tier A and Tier B patients to GDC RNA-seq files
# 2. Identify patients with multiple primary tumor samples
# 3. Identify samples with multiple RNA-seq files
# 4. Inventory available normal tissue samples
# 5. Estimate storage requirements
# 6. Save reproducible candidate file inventories
#
# IMPORTANT:
# No final RNA-seq file selection is performed.
# No molecular expression data is downloaded.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import json

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

RESULTS_DIR = PROJECT_ROOT / "results" / "tables"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

INVENTORY_PATH = (
    METADATA_DIR /
    "tcga_rnaseq_file_inventory.csv"
)

TIER_A_PATH = (
    METADATA_DIR /
    "tcga_tier_a_published_mss.csv"
)

TIER_B_PATH = (
    METADATA_DIR /
    "tcga_tier_b_computational_low_msi.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.10 — RNA-SEQ SAMPLE SELECTION AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load research metadata
# ------------------------------------------------------------

required_files = [
    INVENTORY_PATH,
    TIER_A_PATH,
    TIER_B_PATH
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required file missing: {path}"
        )

inventory = pd.read_csv(
    INVENTORY_PATH,
    dtype=str,
    keep_default_na=False
)

tier_a = pd.read_csv(
    TIER_A_PATH,
    dtype=str,
    keep_default_na=False
)

tier_b = pd.read_csv(
    TIER_B_PATH,
    dtype=str,
    keep_default_na=False
)

print("\nMetadata loaded successfully.")

print(
    "Total RNA-seq files:",
    inventory["file_id"].nunique()
)

print("Tier A patients:", len(tier_a))

print("Tier B patients:", len(tier_b))

# ------------------------------------------------------------
# 3. Validate cohort identifiers
# ------------------------------------------------------------

if tier_a["patientId"].duplicated().any():
    raise RuntimeError(
        "Duplicate patients detected in Tier A."
    )

if tier_b["patientId"].duplicated().any():
    raise RuntimeError(
        "Duplicate patients detected in Tier B."
    )

tier_a_ids = set(tier_a["patientId"])

tier_b_ids = set(tier_b["patientId"])

if tier_a_ids & tier_b_ids:
    raise RuntimeError(
        "Tier A and Tier B contain overlapping patients."
    )

if inventory["file_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate file IDs in inventory. "
        "Review metadata before proceeding."
    )

print("\nCohort validation: PASSED")

# ------------------------------------------------------------
# 4. Construct patient-level cohort lookup
# ------------------------------------------------------------

tier_a_lookup = tier_a[
    ["patientId", "gdc_project"]
].copy()

tier_a_lookup["cohort_tier"] = "Tier A"

tier_b_lookup = tier_b[
    ["patientId", "gdc_project"]
].copy()

tier_b_lookup["cohort_tier"] = "Tier B"

cohort_lookup = pd.concat(
    [
        tier_a_lookup,
        tier_b_lookup
    ],
    ignore_index=True
)

cohort_lookup = cohort_lookup.rename(
    columns={
        "patientId": "patient_id",
        "gdc_project": "expected_project"
    }
)

# ------------------------------------------------------------
# 5. Prepare primary tumor RNA-seq inventory
# ------------------------------------------------------------

primary = inventory[
    inventory["sample_type"] == "Primary Tumor"
].copy()

primary = primary.merge(
    cohort_lookup,
    on="patient_id",
    how="inner",
    validate="many_to_one"
)

# Verify GDC project correspondence.

project_mismatch = (
    primary["project"]
    != primary["expected_project"]
)

if project_mismatch.any():
    raise RuntimeError(
        "GDC project mismatch detected."
    )

# Every eligible patient must have a primary tumor file.

matched_patients = set(
    primary["patient_id"]
)

expected_patients = (
    tier_a_ids | tier_b_ids
)

missing_patients = (
    expected_patients - matched_patients
)

if missing_patients:
    raise RuntimeError(
        f"{len(missing_patients)} cohort patients "
        "have no matching primary tumor RNA-seq file."
    )

print("\nPrimary tumor matching: PASSED")

# ------------------------------------------------------------
# 6. Convert file sizes
# ------------------------------------------------------------

primary["file_size_bytes"] = pd.to_numeric(
    primary["file_size_bytes"],
    errors="coerce"
)

if primary["file_size_bytes"].isna().any():
    raise RuntimeError(
        "Missing file sizes detected."
    )

primary["file_size_mb"] = (
    primary["file_size_bytes"]
    / (1024 ** 2)
)

# ------------------------------------------------------------
# 7. Cohort-specific RNA-seq inventory
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("COHORT RNA-SEQ FILE AVAILABILITY")
print("-" * 65)

cohort_summary = (
    primary.groupby(
        ["project", "cohort_tier"]
    )
    .agg(
        patients=("patient_id", "nunique"),
        samples=("sample_id", "nunique"),
        files=("file_id", "nunique"),
        total_size_mb=("file_size_mb", "sum")
    )
    .reset_index()
)

cohort_summary["total_size_mb"] = (
    cohort_summary["total_size_mb"].round(2)
)

print("\n")
print(
    cohort_summary.to_string(index=False)
)

# ------------------------------------------------------------
# 8. Identify patients with multiple tumor samples
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MULTIPLE PRIMARY TUMOR SAMPLE AUDIT")
print("-" * 65)

patient_sample_counts = (
    primary.groupby(
        ["project", "cohort_tier", "patient_id"]
    )["sample_id"]
    .nunique()
    .reset_index(name="n_samples")
)

multi_sample_patients = (
    patient_sample_counts[
        patient_sample_counts["n_samples"] > 1
    ]
    .copy()
)

print(
    "\nPatients with multiple primary tumor samples:",
    len(multi_sample_patients)
)

if not multi_sample_patients.empty:

    print("\nPatients requiring sample selection review:")

    print(
        multi_sample_patients
        .sort_values(
            ["project", "cohort_tier", "patient_id"]
        )
        .to_string(index=False)
    )

else:

    print(
        "No patients with multiple primary tumor samples."
    )

# ------------------------------------------------------------
# 9. Identify samples with multiple RNA-seq files
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MULTIPLE RNA-SEQ FILE AUDIT")
print("-" * 65)

sample_file_counts = (
    primary.groupby(
        [
            "project",
            "cohort_tier",
            "patient_id",
            "sample_id"
        ]
    )["file_id"]
    .nunique()
    .reset_index(name="n_files")
)

multi_file_samples = (
    sample_file_counts[
        sample_file_counts["n_files"] > 1
    ]
    .copy()
)

print(
    "\nSamples with multiple RNA-seq files:",
    len(multi_file_samples)
)

if not multi_file_samples.empty:

    print("\nSamples requiring file selection review:")

    print(
        multi_file_samples
        .sort_values(
            ["project", "cohort_tier", "patient_id"]
        )
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 10. Generate detailed duplicate file audit
# ------------------------------------------------------------

duplicate_file_details = primary.merge(
    multi_file_samples[
        ["sample_id"]
    ],
    on="sample_id",
    how="inner"
)

duplicate_file_details = (
    duplicate_file_details
    .sort_values(
        ["project", "patient_id", "sample_id", "file_id"]
    )
)

if not duplicate_file_details.empty:

    print("\nDetailed duplicate file information:")

    display_columns = [
        "project",
        "cohort_tier",
        "patient_id",
        "sample_id",
        "file_id",
        "file_name",
        "file_size_mb"
    ]

    print(
        duplicate_file_details[
            display_columns
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# 11. Inventory normal tissue samples
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("NORMAL TISSUE SAMPLE AVAILABILITY")
print("-" * 65)

normal = inventory[
    inventory["sample_type"] == "Solid Tissue Normal"
].copy()

normal["file_size_bytes"] = pd.to_numeric(
    normal["file_size_bytes"],
    errors="coerce"
)

if normal["file_size_bytes"].isna().any():
    raise RuntimeError(
        "Missing normal tissue file sizes."
    )

normal["file_size_mb"] = (
    normal["file_size_bytes"]
    / (1024 ** 2)
)

normal_summary = (
    normal.groupby("project")
    .agg(
        patients=("patient_id", "nunique"),
        samples=("sample_id", "nunique"),
        files=("file_id", "nunique"),
        total_size_mb=("file_size_mb", "sum")
    )
    .reset_index()
)

normal_summary["total_size_mb"] = (
    normal_summary["total_size_mb"].round(2)
)

print("\n")
print(
    normal_summary.to_string(index=False)
)

# ------------------------------------------------------------
# 12. Investigate paired normal availability
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("TUMOR / NORMAL PATIENT OVERLAP")
print("-" * 65)

normal_patients = set(
    normal["patient_id"]
)

for project in ["TCGA-COAD", "TCGA-READ"]:

    for tier_name, tier_ids in [
        ("Tier A", tier_a_ids),
        ("Tier B", tier_b_ids)
    ]:

        project_patients = set(
            primary.loc[
                (primary["project"] == project)
                & (primary["cohort_tier"] == tier_name),
                "patient_id"
            ]
        )

        paired_patients = (
            project_patients & normal_patients
        )

        print(
            f"{project} | {tier_name} | "
            f"Patients with available normal tissue: "
            f"{len(paired_patients)}"
        )

print("\nNOTE:")
print(
    "Patient overlap does not guarantee identical "
    "sample processing or a complete paired analysis."
)

# ------------------------------------------------------------
# 13. Storage requirement estimate
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ESTIMATED RNA-SEQ STORAGE REQUIREMENTS")
print("-" * 65)

tumor_bytes = primary["file_size_bytes"].sum()

normal_bytes = normal["file_size_bytes"].sum()

combined_bytes = tumor_bytes + normal_bytes

tumor_gb = tumor_bytes / (1024 ** 3)

normal_gb = normal_bytes / (1024 ** 3)

combined_gb = combined_bytes / (1024 ** 3)

print(f"\nCandidate tumor files: {tumor_gb:.3f} GB")

print(f"Normal tissue files:   {normal_gb:.3f} GB")

print(f"Combined total:        {combined_gb:.3f} GB")

print("\nIMPORTANT:")
print(
    "This estimate includes all candidate files, "
    "including those requiring duplicate review."
)

print(
    "It excludes temporary processing overhead "
    "and additional downstream datasets."
)

# ------------------------------------------------------------
# 14. Save candidate file inventories
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING RESEARCH INVENTORIES")
print("-" * 65)

tumor_path = (
    METADATA_DIR /
    "tcga_candidate_tumor_rnaseq_files.csv"
)

normal_path = (
    METADATA_DIR /
    "tcga_candidate_normal_rnaseq_files.csv"
)

multi_sample_path = (
    METADATA_DIR /
    "tcga_multiple_tumor_samples_review.csv"
)

multi_file_path = (
    METADATA_DIR /
    "tcga_multiple_rnaseq_files_review.csv"
)

duplicate_details_path = (
    METADATA_DIR /
    "tcga_duplicate_file_details.csv"
)

summary_path = (
    RESULTS_DIR /
    "tcga_rnaseq_candidate_file_summary.csv"
)

primary.to_csv(
    tumor_path,
    index=False
)

normal.to_csv(
    normal_path,
    index=False
)

multi_sample_patients.to_csv(
    multi_sample_path,
    index=False
)

multi_file_samples.to_csv(
    multi_file_path,
    index=False
)

duplicate_file_details.to_csv(
    duplicate_details_path,
    index=False
)

cohort_summary.to_csv(
    summary_path,
    index=False
)

# ------------------------------------------------------------
# 15. Save audit provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.10",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "NCI Genomic Data Commons",

    "workflow": "STAR - Counts",

    "tier_a_patients": len(tier_a_ids),

    "tier_b_patients": len(tier_b_ids),

    "candidate_tumor_files": int(
        primary["file_id"].nunique()
    ),

    "candidate_normal_files": int(
        normal["file_id"].nunique()
    ),

    "patients_with_multiple_tumor_samples": int(
        len(multi_sample_patients)
    ),

    "samples_with_multiple_rnaseq_files": int(
        len(multi_file_samples)
    ),

    "candidate_storage_gb": round(
        float(combined_gb), 4
    ),

    "final_file_selection_performed": False,

    "molecular_data_downloaded": False,

    "limitations": [
        "Multiple tumor samples require review.",
        "Multiple RNA-seq files require review.",
        "Normal samples are not automatically matched controls.",
        "Tier B is computationally low-MSI, not confirmed MSS.",
        "Candidate file inventory is not a finalized download manifest."
    ]
}

provenance_path = (
    METADATA_DIR /
    "tcga_rnaseq_sample_audit_provenance.json"
)

with open(provenance_path, "w") as f:

    json.dump(
        provenance,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 16. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.10 SUMMARY")
print("=" * 65)

print("\nCandidate tumor patients:", len(expected_patients))

print(
    "Candidate tumor RNA-seq files:",
    primary["file_id"].nunique()
)

print(
    "Candidate normal RNA-seq files:",
    normal["file_id"].nunique()
)

print(
    "\nPatients requiring multiple-sample review:",
    len(multi_sample_patients)
)

print(
    "Samples requiring multiple-file review:",
    len(multi_file_samples)
)

print("\nFiles saved:")

print("1. tcga_candidate_tumor_rnaseq_files.csv")
print("2. tcga_candidate_normal_rnaseq_files.csv")
print("3. tcga_multiple_tumor_samples_review.csv")
print("4. tcga_multiple_rnaseq_files_review.csv")
print("5. tcga_duplicate_file_details.csv")
print("6. tcga_rnaseq_candidate_file_summary.csv")
print("7. tcga_rnaseq_sample_audit_provenance.json")

print("\nNo expression matrices downloaded.")

print("No duplicate files automatically selected.")

print("\nPATCH 1.10 COMPLETE")

print("=" * 65)